In [1]:
import sys
import pathlib
from pathlib import Path


p = Path.cwd().resolve()
while p != p.parent and not (p / "modules").is_dir():
    p = p.parent
sys.path.insert(0, str(p))



import pandas as pd
from tabulate import tabulate
import time

from modules.utils.query import query
from modules.utils.dates import to_datetime, add_date_parts
from modules.utils.progress import step
from modules.analytics.grouping import group_unique, group_sum, group_average
from modules.analytics.penetration import row_penetration

In [2]:
def load_prm_data(start: str, end: str) -> pd.DataFrame:
    """ 
    Load PRM Data where Billing PRM = 1, for time period 
    
    Parameters
    ----------
    start: str
        Start of window (ISO format)
    end: str
        End of window (ISO format)
    
     Returns
    ----------
    pd.DataFrame
        Dataframe of PRM Data with columns:
        ['Job ID', 'Passenger ID', 'Operation Date', 'Vehicle Type', 'Operation Date_day', 'Operation Date_month', 'Operation Date_year']
    """

    df = query(
        table="PRM.CompletedServicesByJob",
        columns = [
            "RequestID AS [Job ID]",
            "PassengerID AS [Passenger ID]",
            "ArrDep AS [A/D]",
            "AirlineCode_IATA AS [Airline Code]",
            "FlightNumber AS [Flight Number]",
            "ScheduledDateTime_Local AS [Scheduled Flight DT]",
        ],
        where = ["BillingPRM = 1",
                 "ScheduledDateTime_Local >= :start_op",
                 "ScheduledDateTime_Local < :end_op",
        ],
        params= {"start_op": start, "end_op": end},
        query_option = "OPTION (RECOMPILE)",
    )
    
    df = to_datetime(df, "Scheduled Flight DT")
    df = add_date_parts(df, "Scheduled Flight DT", day=True, year=True)

    df["Flight Number"] = df["Flight Number"].astype(str).str.lstrip("0")

    return df

def load_flight_data(start:str, end:str) -> pd.DataFrame:
    """
    Load flight historical data for time period
    
    Parameters
    ----------
    start: str
        Start of window (ISO format)
    end: str
        End of window (ISO format)
    
     Returns
    ----------
    pd.DataFrame
        Dataframe of PRM Data with columns:
        ['Actual DateTime', 'Pax', 'Actual DateTime_month', 'Actual DateTime_year', ''Actual DateTime_day']
    
    """
    df = query(
        table="Eal.FlightPerformance",
        columns = [
            "ScheduledDateTime_Local AS [Scheduled Flight DT]",
            "ArrDeptureCode AS [A/D]",
            "FlightNumber AS [Flight Number]",
            "AirlineCode_IATA AS [Airline Code]",
            "Passengers AS [Pax]",
            "AirportCountryCode AS [Airport Country Code]",

        ],
        where = ["IsPassengerFlight = 1",
                 "ScheduledDateTime_Local >= :start_op",
                 "ScheduledDateTime_Local < :end_op",
        ],
        params= {"start_op": start, "end_op": end},
        query_option = "OPTION (RECOMPILE)",

    )
    df = to_datetime(df, "Scheduled Flight DT")
    df = add_date_parts(df, "Scheduled Flight DT", day=True, year=True)

    df["Flight Number"] = df["Flight Number"].astype(str).str.lstrip("0")
    
    return df

In [3]:
FLIGHT_KEY = ["Day", "Airline Code", "Flight Number"]


def _ensure_date_parts(df: pd.DataFrame, dt_col: str = "Scheduled Flight DT") -> pd.DataFrame:
    x = df.copy()
    x = to_datetime(x, dt_col)
    if "Day" not in x.columns:
        x = add_date_parts(x, dt_col, day=True)
    return x


def build_flight_level_dataset(
    prm_df: pd.DataFrame,
    flight_df: pd.DataFrame,
    *,
    dt_col: str = "Scheduled Flight DT",
    passenger_id_col: str = "Passenger ID",
    pax_col: str = "Pax",
    country_col: str = "Airport Country Code",
) -> pd.DataFrame:
    prm = _ensure_date_parts(prm_df, dt_col=dt_col)
    flt = _ensure_date_parts(flight_df, dt_col=dt_col)

    # light normalisation (doesn't conflict with your loader stripping)
    prm["Airline Code"] = prm["Airline Code"].astype(str).str.strip().str.upper()
    flt["Airline Code"] = flt["Airline Code"].astype(str).str.strip().str.upper()

    prm["Flight Number"] = prm["Flight Number"].astype(str).str.strip()
    flt["Flight Number"] = flt["Flight Number"].astype(str).str.strip()

    prm_by_flight = (
        group_unique(prm, by_cols=FLIGHT_KEY, id_col=passenger_id_col)
        .rename(columns={"Unique Count": "PRM Count"})
    )

    pax_by_flight = group_sum(flt, by_cols=FLIGHT_KEY, value_col=pax_col, out_col="Total Pax")

    if country_col in flt.columns:
        country_lookup = (
            flt.groupby(FLIGHT_KEY, dropna=False)[country_col]
               .first()
               .reset_index()
        )
        pax_by_flight = pax_by_flight.merge(country_lookup, on=FLIGHT_KEY, how="left")

    merged = pax_by_flight.merge(prm_by_flight, on=FLIGHT_KEY, how="left")
    merged["PRM Count"] = merged["PRM Count"].fillna(0).astype(int)

    # per-flight penetration retained (harmless + useful for debugging)
    merged = row_penetration(
        merged,
        numerator_col="PRM Count",
        denominator_col="Total Pax",
        out_col="Penetration Rate",
    )

    d = pd.to_datetime(merged["Day"], errors="coerce")
    merged["Month Start"] = d.dt.to_period("M").dt.to_timestamp()

    return merged


def filter_last_3_years_months(df: pd.DataFrame, month_col: str = "Month Start") -> pd.DataFrame:
    x = df.copy()
    max_m = pd.to_datetime(x[month_col]).max()
    cutoff = (max_m.to_period("M") - 35).to_timestamp()
    return x[x[month_col] >= cutoff].copy()


def monthly_totals_with_overall_and_yoy(
    flight_level_df: pd.DataFrame,
    group_col: str,
) -> pd.DataFrame:
    x = flight_level_df.copy()

    grp_by = ["Month Start", group_col]
    grp_pax = group_sum(x, by_cols=grp_by, value_col="Total Pax", out_col="Group Total Pax")
    grp_prm = group_sum(x, by_cols=grp_by, value_col="PRM Count", out_col="Group Total PRM")
    grp = grp_pax.merge(grp_prm, on=grp_by, how="left")

    all_by = ["Month Start"]
    all_pax = group_sum(x, by_cols=all_by, value_col="Total Pax", out_col="All Total Pax")

    out = grp.merge(all_pax, on="Month Start", how="left")

    # Group pen rate = group PRM / group pax
    out["Group Penetration Rate"] = (
        out["Group Total PRM"].astype(float)
           .divide(out["Group Total Pax"].astype(float))
           .where(out["Group Total Pax"].ne(0))
    )

    # All pen rate (as you defined) = group PRM / ALL pax
    out["All Penetration Rate"] = (
        out["Group Total PRM"].astype(float)
           .divide(out["All Total Pax"].astype(float))
           .where(out["All Total Pax"].ne(0))
    )

    # YoY deltas (no MoM)
    out = out.sort_values([group_col, "Month Start"])
    out["YoY Δ (Group Pen Rate)"] = out.groupby(group_col)["Group Penetration Rate"].diff(12)
    out["YoY Δ (All Pen Rate)"] = out.groupby(group_col)["All Penetration Rate"].diff(12)

    return out


def _fmt_int(v):
    return f"{int(v):,}" if pd.notnull(v) else ""


def _fmt_pct(v):
    return f"{v:.4%}" if pd.notnull(v) else ""


def wide_table_for_one_group(
    monthly_df: pd.DataFrame,
    group_col: str,
    group_value,
) -> pd.DataFrame:
    one = monthly_df[monthly_df[group_col] == group_value].copy()
    one = one.sort_values("Month Start")
    one["Month"] = pd.to_datetime(one["Month Start"]).dt.strftime("%Y-%m")

    metrics = {
        "Total PRM (group)": one.set_index("Month")["Group Total PRM"].map(_fmt_int).to_dict(),
        "Total Pax (group)": one.set_index("Month")["Group Total Pax"].map(_fmt_int).to_dict(),
        "Total Pax (all)": one.set_index("Month")["All Total Pax"].map(_fmt_int).to_dict(),
        "Pen Rate (group PRM / group pax)": one.set_index("Month")["Group Penetration Rate"].map(_fmt_pct).to_dict(),
        "Pen Rate (group PRM / all pax)": one.set_index("Month")["All Penetration Rate"].map(_fmt_pct).to_dict(),
        "YoY Δ (group pen rate)": one.set_index("Month")["YoY Δ (Group Pen Rate)"].map(_fmt_pct).to_dict(),
        "YoY Δ (all pen rate)": one.set_index("Month")["YoY Δ (All Pen Rate)"].map(_fmt_pct).to_dict(),
    }

    wide = pd.DataFrame(metrics).T
    wide.index.name = "Metric"
    return wide


def _group_order(monthly_df: pd.DataFrame, group_col: str) -> list:
    x = monthly_df.copy()
    x[group_col] = x[group_col].where(x[group_col].notna(), "Unknown")
    return (
        x.groupby(group_col, dropna=False)["Group Total PRM"]
         .sum()
         .sort_values(ascending=False)
         .index
         .tolist()
    )


def _write_group_blocks_to_sheet(writer, sheet_name: str, monthly_df: pd.DataFrame, group_col: str) -> None:
    order = _group_order(monthly_df, group_col)
    row = 0

    for g in order:
        header = pd.DataFrame({group_col: [g]})
        header.to_excel(writer, sheet_name=sheet_name, index=False, startrow=row)
        row += 2

        wide = wide_table_for_one_group(monthly_df, group_col, g)
        out = wide.copy()
        out.insert(0, "Metric", out.index)
        out = out.reset_index(drop=True)

        out.to_excel(writer, sheet_name=sheet_name, index=False, startrow=row)
        row += len(out) + 3


def export_penetration_excel(
    airline_monthly: pd.DataFrame,
    country_monthly: pd.DataFrame | None,
    out_path: str = "outputs/prm/penetration.xlsx",
) -> str:
    out_file = Path(out_path)
    out_file.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
        _write_group_blocks_to_sheet(writer, "Airline", airline_monthly, "Airline Code")

        if country_monthly is not None and len(country_monthly) > 0:
            _write_group_blocks_to_sheet(writer, "Country", country_monthly, "Airport Country Code")
        else:
            pd.DataFrame({"Info": ["No country data present."]}).to_excel(writer, sheet_name="Country", index=False)

    return str(out_file)



def negative_yoy_list_year(monthly_df: pd.DataFrame, group_col: str, yoy_col: str, year: int):
    x = monthly_df.copy()
    x["Month Start"] = pd.to_datetime(x["Month Start"], errors="coerce")
    x = x[(x["Month Start"].dt.year == year) & (pd.to_numeric(x[yoy_col], errors="coerce") < 0)].copy()
    x["Month"] = x["Month Start"].dt.strftime("%Y-%m")
    x[group_col] = x[group_col].where(x[group_col].notna(), "Unknown")

    by_month = (
        x.groupby("Month")[group_col]
         .apply(lambda s: sorted(set(s.astype(str))))
         .to_dict()
    )

    by_group = (
        x.groupby(group_col)["Month"]
         .apply(lambda s: sorted(set(s.astype(str))))
         .to_dict()
    )

    return x, by_month, by_group




def print_negative_yoy_year(monthly_df: pd.DataFrame, group_col: str, yoy_col: str, title: str, year: int):
    x, by_month, by_group = negative_yoy_list_year(monthly_df, group_col, yoy_col, year)

    print("\n" + title)
    if x.empty:
        print(f"No negative YoY months in {year}.")
        return

    month_rows = [{"Month": m, "Count": len(v), group_col: ", ".join(v)} for m, v in sorted(by_month.items())]
    print(f"\nNegative YoY by month ({year}):")
    print(tabulate(month_rows, headers="keys", tablefmt="github", showindex=False))

    group_rows = [{"Entity": g, "Count": len(ms), "Months": ", ".join(ms)} for g, ms in sorted(by_group.items())]
    print(f"\nEntities with at least one negative YoY month ({year}):")
    print(tabulate(group_rows, headers="keys", tablefmt="github", showindex=False))



def run_all(
    start: str,
    end: str,
    *,
    top_n_airlines: int | None = None,
    top_n_countries: int | None = None,
    out_path: str = "outputs/prm/penetration.xlsx",
) -> str:
    prm = load_prm_data(start, end)
    flt = load_flight_data(start, end)

    flight_level = build_flight_level_dataset(prm, flt)

    airline_monthly = monthly_totals_with_overall_and_yoy(flight_level, "Airline Code")
    airline_monthly = filter_last_3_years_months(airline_monthly, "Month Start")

    if top_n_airlines is not None:
        keep = _group_order(airline_monthly, "Airline Code")[:top_n_airlines]
        airline_monthly = airline_monthly[airline_monthly["Airline Code"].isin(keep)].copy()

    country_monthly = None
    if "Airport Country Code" in flight_level.columns:
        country_monthly = monthly_totals_with_overall_and_yoy(flight_level, "Airport Country Code")
        country_monthly = filter_last_3_years_months(country_monthly, "Month Start")

        if top_n_countries is not None:
            keep = _group_order(country_monthly, "Airport Country Code")[:top_n_countries]
            country_monthly = country_monthly[country_monthly["Airport Country Code"].isin(keep)].copy()

    # ---- PRINT NEGATIVE YOY LISTS FOR 2026 ----
    print_negative_yoy_year(
        airline_monthly,
        group_col="Airline Code",
        yoy_col="YoY Δ (Group Pen Rate)",
        title="Airlines with negative YoY (Group Pen Rate) in 2026",
        year=2026,
    )

    print_negative_yoy_year(
        airline_monthly,
        group_col="Airline Code",
        yoy_col="YoY Δ (All Pen Rate)",
        title="Airlines with negative YoY (All Pen Rate) in 2026",
        year=2026,
    )

    if country_monthly is not None and len(country_monthly) > 0:
        print_negative_yoy_year(
            country_monthly,
            group_col="Airport Country Code",
            yoy_col="YoY Δ (Group Pen Rate)",
            title="Countries with negative YoY (Group Pen Rate) in 2026",
            year=2026,
        )

        print_negative_yoy_year(
            country_monthly,
            group_col="Airport Country Code",
            yoy_col="YoY Δ (All Pen Rate)",
            title="Countries with negative YoY (All Pen Rate) in 2026",
            year=2026,
        )

    return export_penetration_excel(airline_monthly, country_monthly, out_path=out_path)


In [4]:
path = run_all("2023-01-01", "2026-05-01", out_path="outputs/prm/penetration.xlsx")
print(path)



Airlines with negative YoY (Group Pen Rate) in 2026

Negative YoY by month (2026):
| Month   |   Count | Airline Code                                           |
|---------|---------|--------------------------------------------------------|
| 2026-01 |       9 | A3, BY, D8, DS, DY, EC, FR, SK, VY                     |
| 2026-02 |       7 | A3, AY, D8, EC, SN, TK, VY                             |
| 2026-03 |      12 | AP, BY, D8, DY, EI, FR, SN, TO, UA, VY, WK, XQ         |
| 2026-04 |      14 | A3, AC, D8, DY, EI, GR, HU, LM, SK, SN, TO, VY, WK, WS |

Entities with at least one negative YoY month (2026):
| Entity   |   Count | Months                             |
|----------|---------|------------------------------------|
| A3       |       3 | 2026-01, 2026-02, 2026-04          |
| AC       |       1 | 2026-04                            |
| AP       |       1 | 2026-03                            |
| AY       |       1 | 2026-02                            |
| BY       |       2 | 2026